In [5]:
!pip install pandas

In [6]:
import random
import json
import time
import pandas as pd

In [9]:

"1"
event_hub = []

def generar_transaccion():
    tipo = random.choice(["normal", "alto_valor", "invalido"])

    if tipo == "normal":
        return {
            "id": random.randint(1, 1000),
            "comercio": "Tienda A",
            "monto": random.randint(1000, 300000),
            "tipo": "normal"
        }

    if tipo == "alto_valor":
        return {
            "id": random.randint(1, 1000),
            "comercio": "Tienda VIP",
            "monto": random.randint(5000000, 10000000),
            "tipo": "alto_valor"
        }

    return "ERROR_JSON"


def publicar_eventos(n=10):
    for _ in range(n):
        evento = generar_transaccion()
        event_hub.append(evento)
        print("📨 Evento enviado a Event Hub:", evento)

publicar_eventos(10)

📨 Evento enviado a Event Hub: {'id': 725, 'comercio': 'Tienda VIP', 'monto': 7708219, 'tipo': 'alto_valor'}
📨 Evento enviado a Event Hub: {'id': 714, 'comercio': 'Tienda A', 'monto': 267779, 'tipo': 'normal'}
📨 Evento enviado a Event Hub: {'id': 48, 'comercio': 'Tienda VIP', 'monto': 8937227, 'tipo': 'alto_valor'}
📨 Evento enviado a Event Hub: ERROR_JSON
📨 Evento enviado a Event Hub: ERROR_JSON
📨 Evento enviado a Event Hub: ERROR_JSON
📨 Evento enviado a Event Hub: ERROR_JSON
📨 Evento enviado a Event Hub: {'id': 764, 'comercio': 'Tienda A', 'monto': 132312, 'tipo': 'normal'}
📨 Evento enviado a Event Hub: {'id': 479, 'comercio': 'Tienda VIP', 'monto': 7702553, 'tipo': 'alto_valor'}
📨 Evento enviado a Event Hub: {'id': 99, 'comercio': 'Tienda A', 'monto': 104333, 'tipo': 'normal'}


In [11]:
"2"
procesados = []

def validar_transaccion(evento):

    if not isinstance(evento, dict):
        return {"estado": "rechazada", "motivo": "formato invalido"}

    if "monto" not in evento:
        return {"estado": "rechazada", "motivo": "campo faltante"}

    monto = evento["monto"]

    estado = "aprobada"

    # regla antifraude
    if monto < 0:
        estado = "rechazada"

    # paso 3 (alto valor)
    if monto > 5000000:
        estado = "en_revision"

    return {
        "id": evento["id"],
        "comercio": evento.get("comercio", "desconocido"),
        "monto": monto,
        "estado": estado
    }

In [12]:
"3"
service_bus = []

def enrutar_alto_valor(transaccion):
    if transaccion["estado"] == "en_revision":
        service_bus.append(transaccion)
        print("🚨 Enviado a Service Bus (alto valor):", transaccion)

In [13]:
"4"
cosmos_db = []

def guardar_cosmos(transaccion):
    cosmos_db.append(transaccion)
    print("💾 Guardado en Cosmos DB:", transaccion)

In [14]:
"5"
print("\n🚀 INICIANDO PIPELINE PAYFLOW...\n")

for evento in event_hub:

    # PASO 2
    resultado = validar_transaccion(evento)

    # PASO 3
    enrutar_alto_valor(resultado)

    # PASO 4
    guardar_cosmos(resultado)

    procesados.append(resultado)

print("\n✅ PROCESO FINALIZADO")


🚀 INICIANDO PIPELINE PAYFLOW...

🚨 Enviado a Service Bus (alto valor): {'id': 725, 'comercio': 'Tienda VIP', 'monto': 7708219, 'estado': 'en_revision'}
💾 Guardado en Cosmos DB: {'id': 725, 'comercio': 'Tienda VIP', 'monto': 7708219, 'estado': 'en_revision'}
💾 Guardado en Cosmos DB: {'id': 714, 'comercio': 'Tienda A', 'monto': 267779, 'estado': 'aprobada'}
🚨 Enviado a Service Bus (alto valor): {'id': 48, 'comercio': 'Tienda VIP', 'monto': 8937227, 'estado': 'en_revision'}
💾 Guardado en Cosmos DB: {'id': 48, 'comercio': 'Tienda VIP', 'monto': 8937227, 'estado': 'en_revision'}
💾 Guardado en Cosmos DB: {'estado': 'rechazada', 'motivo': 'formato invalido'}
💾 Guardado en Cosmos DB: {'estado': 'rechazada', 'motivo': 'formato invalido'}
💾 Guardado en Cosmos DB: {'estado': 'rechazada', 'motivo': 'formato invalido'}
💾 Guardado en Cosmos DB: {'estado': 'rechazada', 'motivo': 'formato invalido'}
💾 Guardado en Cosmos DB: {'id': 764, 'comercio': 'Tienda A', 'monto': 132312, 'estado': 'aprobada'}
🚨 

In [26]:
event_hub


[{'id': 725, 'comercio': 'Tienda VIP', 'monto': 7708219, 'tipo': 'alto_valor'},
 {'id': 714, 'comercio': 'Tienda A', 'monto': 267779, 'tipo': 'normal'},
 {'id': 48, 'comercio': 'Tienda VIP', 'monto': 8937227, 'tipo': 'alto_valor'},
 'ERROR_JSON',
 'ERROR_JSON',
 'ERROR_JSON',
 'ERROR_JSON',
 {'id': 764, 'comercio': 'Tienda A', 'monto': 132312, 'tipo': 'normal'},
 {'id': 479, 'comercio': 'Tienda VIP', 'monto': 7702553, 'tipo': 'alto_valor'},
 {'id': 99, 'comercio': 'Tienda A', 'monto': 104333, 'tipo': 'normal'}]

In [16]:
procesados

[{'id': 725,
  'comercio': 'Tienda VIP',
  'monto': 7708219,
  'estado': 'en_revision'},
 {'id': 714, 'comercio': 'Tienda A', 'monto': 267779, 'estado': 'aprobada'},
 {'id': 48,
  'comercio': 'Tienda VIP',
  'monto': 8937227,
  'estado': 'en_revision'},
 {'estado': 'rechazada', 'motivo': 'formato invalido'},
 {'estado': 'rechazada', 'motivo': 'formato invalido'},
 {'estado': 'rechazada', 'motivo': 'formato invalido'},
 {'estado': 'rechazada', 'motivo': 'formato invalido'},
 {'id': 764, 'comercio': 'Tienda A', 'monto': 132312, 'estado': 'aprobada'},
 {'id': 479,
  'comercio': 'Tienda VIP',
  'monto': 7702553,
  'estado': 'en_revision'},
 {'id': 99, 'comercio': 'Tienda A', 'monto': 104333, 'estado': 'aprobada'}]

In [17]:
service_bus

[{'id': 725,
  'comercio': 'Tienda VIP',
  'monto': 7708219,
  'estado': 'en_revision'},
 {'id': 48,
  'comercio': 'Tienda VIP',
  'monto': 8937227,
  'estado': 'en_revision'},
 {'id': 479,
  'comercio': 'Tienda VIP',
  'monto': 7702553,
  'estado': 'en_revision'}]

In [18]:
cosmos_db

[{'id': 725,
  'comercio': 'Tienda VIP',
  'monto': 7708219,
  'estado': 'en_revision'},
 {'id': 714, 'comercio': 'Tienda A', 'monto': 267779, 'estado': 'aprobada'},
 {'id': 48,
  'comercio': 'Tienda VIP',
  'monto': 8937227,
  'estado': 'en_revision'},
 {'estado': 'rechazada', 'motivo': 'formato invalido'},
 {'estado': 'rechazada', 'motivo': 'formato invalido'},
 {'estado': 'rechazada', 'motivo': 'formato invalido'},
 {'estado': 'rechazada', 'motivo': 'formato invalido'},
 {'id': 764, 'comercio': 'Tienda A', 'monto': 132312, 'estado': 'aprobada'},
 {'id': 479,
  'comercio': 'Tienda VIP',
  'monto': 7702553,
  'estado': 'en_revision'},
 {'id': 99, 'comercio': 'Tienda A', 'monto': 104333, 'estado': 'aprobada'}]

In [23]:
import pandas as pd
df = pd.DataFrame(procesados)
df

,id,comercio,monto,estado,motivo
0,725.0,Tienda VIP,7708219.0,en_revision,NaN
1,714.0,Tienda A,267779.0,aprobada,NaN
2,48.0,Tienda VIP,8937227.0,en_revision,NaN
3,NaN,NaN,NaN,rechazada,formato invalido
4,NaN,NaN,NaN,rechazada,formato invalido
5,NaN,NaN,NaN,rechazada,formato invalido
6,NaN,NaN,NaN,rechazada,formato invalido
7,764.0,Tienda A,132312.0,aprobada,NaN
8,479.0,Tienda VIP,7702553.0,en_revision,NaN
9,99.0,Tienda A,104333.0,aprobada,NaN
